# Lab 5: Text Representation (TF-IDF and Word2Vec)



In [1]:
%pip install scikit-learn pandas numpy nltk gensim kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import os
import glob

import numpy as np
import pandas as pd
import nltk
import gensim
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## 1. TF-IDF

TF-IDF (term frequency - inverse document frequency) measures how important a word is in a document compared with a collection of documents (a corpus).

$$ \text{TF}(t, d) = \frac{\text{times } t \text{ appears in } d}{\text{total terms in } d} \qquad \text{IDF}(t) = \log \left(\frac{N}{\text{documents containing } t}\right) $$

In [3]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [4]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data)

df_tfidf = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df_tfidf

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


## 2. Cosine similarity

$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

- Same direction -> similarity = 1
- Orthogonal (90 degrees) -> similarity = 0

In [5]:
cosine_similarity_matrix = cosine_similarity(result)

df_cosine = pd.DataFrame(cosine_similarity_matrix, index=data, columns=data)
df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


## 3. Word2Vec

Word2Vec learns word vectors with a small neural network. It has two architectures: CBOW and **Skip-gram** (`sg=1`), which we use here.

First example: the Fake and Real News dataset (`True.csv`), downloaded with `kagglehub`.

In [6]:
import kagglehub

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print("Path to dataset files:", path)

df_news = pd.read_csv(os.path.join(path, "True.csv"), engine="python", on_bad_lines="skip")
df_news.head()

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\clmentbisaillon\fake-and-real-news-dataset\versions\1


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [7]:
news_tokens = [word_tokenize(text) for text in df_news["text"].dropna()]

w2v = gensim.models.Word2Vec(news_tokens, min_count=1, vector_size=100, window=5, sg=1)

In [8]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram :", w2v.wv.similarity("provide", "program"))
print("Words similar to 'program' - Skip Gram :", w2v.wv.most_similar("program"))

Cosine similarity between 'provide' and 'program' - Skip Gram : 0.44436535
Words similar to 'program' - Skip Gram : [('programme', 0.8185299038887024), ('programs', 0.7920355796813965), ('abolishing', 0.6962392926216125), ('bond-buying', 0.6940274238586426), ('arsenal', 0.693483293056488), ('terminating', 0.6786690950393677), ('voucher', 0.6741466522216797), ('nuclear', 0.6724042892456055), ('state-federal', 0.6670158505439758), ('EB-5', 0.6654140949249268)]


# Tasks

### Task 1: Cosine Similarity
Use cosine similarity to find how similar these sentences are.

- 'This is the first document.'
- 'This document is the second document.'
- 'And this is the third one.'
- 'Is this the first document?'

In [9]:
sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]
names = ["Sentence 1", "Sentence 2", "Sentence 3", "Sentence 4"]

vectorizer = TfidfVectorizer()
sentence_vectors = vectorizer.fit_transform(sentences)

similarity = cosine_similarity(sentence_vectors)
pd.DataFrame(similarity, index=names, columns=names).round(3)

,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.000,0.647,0.308,1.000
Sentence 2,0.647,1.000,0.225,0.647
Sentence 3,0.308,0.225,1.000,0.308
Sentence 4,1.000,0.647,0.308,1.000


**Reading the result:** Sentences 1 and 4 contain exactly the same words (only order and punctuation differ), and TF-IDF ignores word order, so their similarity is 1.0. The other pairs share fewer words, so they score lower.

### Task 2: TF-IDF
Use TF-IDF on these sentences to find the important words.

- 'data science is one of the most important fields of science'
- 'this is one of the best data science courses'
- 'data scientists analyze data'

In [10]:
documents = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]
doc_names = ["Doc 1", "Doc 2", "Doc 3"]

tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(documents)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=tfidf_vec.get_feature_names_out(),
                        index=doc_names)
tfidf_df.round(3)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Doc 1,0.000,0.0,0.0,0.190,0.321,0.321,0.244,0.321,0.488,0.244,0.488,0.000,0.244,0.0
Doc 2,0.000,0.4,0.4,0.236,0.000,0.000,0.304,0.000,0.304,0.304,0.304,0.000,0.304,0.4
Doc 3,0.543,0.0,0.0,0.641,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.543,0.000,0.0


In [11]:
# The 3 highest-scoring words of each document
for doc in tfidf_df.index:
    print(doc, tfidf_df.loc[doc].sort_values(ascending=False).head(3).round(3).to_dict())

Doc 1 {'science': 0.488, 'of': 0.488, 'fields': 0.321}
Doc 2 {'best': 0.4, 'courses': 0.4, 'this': 0.4}
Doc 3 {'data': 0.641, 'analyze': 0.543, 'scientists': 0.543}


In [12]:
# Same thing, but ignoring English stop words (the, is, of, ...)
tfidf_vec2 = TfidfVectorizer(stop_words="english")
tfidf_matrix2 = tfidf_vec2.fit_transform(documents)

tfidf_df2 = pd.DataFrame(tfidf_matrix2.toarray(),
                         columns=tfidf_vec2.get_feature_names_out(),
                         index=doc_names)
display(tfidf_df2.round(3))

for doc in tfidf_df2.index:
    print(doc, tfidf_df2.loc[doc].sort_values(ascending=False).head(3).round(3).to_dict())

,analyze,best,courses,data,fields,important,science,scientists
Doc 1,0.000,0.000,0.000,0.274,0.463,0.463,0.704,0.000
Doc 2,0.000,0.584,0.584,0.345,0.000,0.000,0.445,0.000
Doc 3,0.543,0.000,0.000,0.641,0.000,0.000,0.000,0.543


Doc 1 {'science': 0.704, 'fields': 0.463, 'important': 0.463}
Doc 2 {'best': 0.584, 'courses': 0.584, 'science': 0.445}
Doc 3 {'data': 0.641, 'analyze': 0.543, 'scientists': 0.543}


**Reading the result:** the highest scores mark words that are frequent in one document but rare in the others. Words that appear in many documents (like *data* or *science*) get lower weights.

### Task 3: Word2Vec on the Simpsons dataset

This needs `simpsons_script_lines.csv`. Put it **in the same folder as this notebook** (the cell below also searches the `kagglehub` cache and tries to download it if it is missing).

In [13]:
def find_simpsons_csv():
    # 1) next to the notebook
    if os.path.exists("simpsons_script_lines.csv"):
        return "simpsons_script_lines.csv"
    # 2) anywhere in the kagglehub cache
    cache = os.path.join(os.path.expanduser("~"), ".cache", "kagglehub")
    hits = glob.glob(os.path.join(cache, "**", "simpsons_script_lines.csv"), recursive=True)
    if hits:
        return hits[0]
    # 3) try to download it
    try:
        folder = kagglehub.dataset_download("prashant111/the-simpsons-dataset")
        hits = glob.glob(os.path.join(folder, "**", "simpsons_script_lines.csv"), recursive=True)
        if hits:
            return hits[0]
    except Exception as e:
        print("Download failed:", e)
    raise FileNotFoundError(
        "simpsons_script_lines.csv not found. Download it from Kaggle and put it in the same folder as this notebook."
    )

simpsons_path = find_simpsons_csv()
print("Using:", simpsons_path)

simpsons = pd.read_csv(simpsons_path, low_memory=False, on_bad_lines="skip")
print(simpsons.shape)

100%|██████████| 8.70M/8.70M [00:01<00:00, 5.25MB/s]

Extracting files...


Using: C:\Users\User\.cache\kagglehub\datasets\prashant111\the-simpsons-dataset\versions\1\simpsons_script_lines.csv
(158271, 13)


In [14]:
# Use the 'spoken_words' column and drop empty lines
simpsons = simpsons.dropna(subset=["spoken_words"])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

simpsons["clean_words"] = simpsons["spoken_words"].apply(clean_text)

tokens = [word_tokenize(line) for line in simpsons["clean_words"]]
print("Number of lines:", len(tokens))
print("Example:", tokens[0])

Number of lines: 132112
Example: ['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


In [15]:
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
print("Vocabulary size:", len(Skip_gram_model.wv))

Vocabulary size: 44319


### Task 4: `wv.most_similar()`

Find the words similar to **homer**, **marge** and **bart**.

In [16]:
for name in ["homer", "marge", "bart"]:
    print(name, ":", Skip_gram_model.wv.most_similar(name))
    print()

homer : [('abe', 0.8836971521377563), ('marge', 0.8521810173988342), ('bartholomew', 0.8407907485961914), ('eliza', 0.8313343524932861), ('bart', 0.8312869071960449), ('jay', 0.8305270075798035), ('grampa', 0.8288335204124451), ('apu', 0.8173660039901733), ('jessica', 0.8165459632873535), ('midge', 0.8134046196937561)]

marge : [('homer', 0.8521810173988342), ('abe', 0.8420118689537048), ('sweetheart', 0.8351919651031494), ('sweetie', 0.8313114047050476), ('apu', 0.8209885954856873), ('honey', 0.8191977739334106), ('lisa', 0.8179202079772949), ('midge', 0.8176637291908264), ('lurleen', 0.8094707727432251), ('becky', 0.8088026642799377)]

bart : [('milhouse', 0.8666695952415466), ('lisa', 0.8632697463035583), ('jessica', 0.8541870713233948), ('abe', 0.8360871076583862), ('janey', 0.8353657722473145), ('laddie', 0.8341402411460876), ('grampa', 0.8336193561553955), ('homer', 0.8312868475914001), ('eliza', 0.8296932578086853), ('nelson', 0.828029215335846)]



Word2Vec starts from random weights, so the exact words and scores change a little from run to run. Characters usually appear in similar contexts, so other character names tend to rank high.

### Task 5: `wv.doesnt_match()`

1. Which of 'jimbo', 'milhouse', 'kearney' does not belong?
2. Odd one among 'nelson', 'bart', 'milhouse'.
3. Odd one among 'homer', 'patty', 'selma'.

*Hint: pass the words as a list.*

In [17]:
print(Skip_gram_model.wv.doesnt_match(["jimbo", "milhouse", "kearney"]))
print(Skip_gram_model.wv.doesnt_match(["nelson", "bart", "milhouse"]))
print(Skip_gram_model.wv.doesnt_match(["homer", "patty", "selma"]))

milhouse
nelson
homer


`doesnt_match()` returns the word whose vector is farthest from the average vector of the list. The answers can vary slightly between runs.